<a href="https://colab.research.google.com/github/geek4s/legalresearchtool/blob/main/legalresearchRAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ─── CELL 1: Install Dependencies ───────────────────────────────────────────
!pip install -q groq chromadb sentence-transformers PyPDF2 python-docx numpy

In [ ]:
# ─── CELL 2: Imports & Configuration ────────────────────────────────────────
import os
from pathlib import Path
from typing import Optional

from groq import Groq
import chromadb
from chromadb.utils import embedding_functions
import PyPDF2
import numpy as np

# ── Set your Groq API key ─────────────────────────────────────────────────────
import getpass
os.environ["GROQ_API_KEY"] = getpass.getpass("Paste your Groq API key: ")

groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL = "llama-3.1-8b-instant"

# ── ChromaDB collections ──────────────────────────────────────────────────────
chroma_client = chromadb.Client()
embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

CASE_LAW_COLLECTION   = chroma_client.get_or_create_collection("case_law",   embedding_function=embed_fn)
COMPLIANCE_COLLECTION = chroma_client.get_or_create_collection("compliance", embedding_function=embed_fn)
REGULATORY_COLLECTION = chroma_client.get_or_create_collection("regulatory", embedding_function=embed_fn)

print("✅ Groq client and ChromaDB collections ready")

Paste your Groq API key: ··········
✅ Groq client and ChromaDB collections ready


In [ ]:
# ─── CELL 3: Create Sample Legal Documents ───────────────────────────────────
sample_docs = {
    "roe_v_wade.txt": """CASE: Roe v. Wade (1973)
Citation: 410 U.S. 113
Court: Supreme Court of the United States
Decision: The Court ruled that the Constitution protects a woman's liberty to
choose to have an abortion. The right to privacy under the Due Process Clause
extends to a woman's decision to terminate a pregnancy.
Precedent: Established trimester framework for abortion regulation.
Related Cases: Planned Parenthood v. Casey (1992), Dobbs v. Jackson (2022)
Key Principle: Constitutional privacy rights; substantive due process.""",

    "gdpr_compliance.txt": """GDPR COMPLIANCE POLICY - Version 3.1
Effective Date: 2024-01-01
Regulatory Reference: EU Regulation 2016/679
Data Retention: Personal data must not be kept longer than necessary.
Default retention: 36 months. Financial records: 7 years (tax compliance).
Right to Erasure: Individuals may request deletion under Article 17 within 30 days.
Data Breach Notification: Supervisory authority must be notified within 72 hours (Article 33).
Penalties: Up to EUR 20 million or 4% of global annual turnover, whichever is higher.""",

    "sec_ruling_2024.txt": """REGULATORY UPDATE - SEC Final Rule
Title: Cybersecurity Risk Management, Strategy, Governance, and Incident Disclosure
Effective: December 2023
Key Requirements:
1. Public companies must disclose material cybersecurity incidents within 4 business days.
2. Annual disclosure of cybersecurity risk management processes required in Form 10-K.
3. Board oversight of cybersecurity risks must be documented.
Affected Entities: All SEC-registered public companies.
Compliance Deadline: Large accelerated filers: Dec 18, 2023. All others: June 15, 2024.
Penalties: Non-disclosure is subject to SEC enforcement action.""",

    "employment_discrimination.txt": """INTERNAL LEGAL BRIEF
Matter: Johnson v. TechCorp Employment Discrimination
Date: February 2024
Summary: Client alleges wrongful termination based on race under Title VII.
Key Precedents:
- McDonnell Douglas Corp. v. Green, 411 U.S. 792 (1973): burden-shifting framework
- Texas Dept. of Community Affairs v. Burdine, 450 U.S. 248 (1981): employer must show
  legitimate non-discriminatory reason for termination.
Recommended Action: File EEOC charge within 300-day statute of limitations.
Preserve all electronic communications per litigation hold.""",
}

for filename, content in sample_docs.items():
    Path(filename).write_text(content, encoding="utf-8")

print(f"✅ {len(sample_docs)} sample legal documents created")

✅ 4 sample legal documents created


In [ ]:
# ─── CELL 4: Document Ingestion ──────────────────────────────────────────────
def extract_text_from_pdf(file_path: str) -> str:
    with open(file_path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        return "\n".join(page.extract_text() or "" for page in reader.pages)

def load_document(file_path: str) -> str:
    ext = Path(file_path).suffix.lower()
    if ext == ".pdf":
        return extract_text_from_pdf(file_path)
    elif ext in (".txt", ".md"):
        return Path(file_path).read_text(encoding="utf-8", errors="ignore")
    else:
        raise ValueError(f"Unsupported file type: {ext}")

def sanitize_text(text: str) -> str:
    # Replace common non-ASCII characters with ASCII equivalents
    text = text.replace('—', '-')  # Em-dash
    text = text.replace('…', '...') # Ellipsis
    # Add more replacements as needed, or use a more general normalization:
    # import unicodedata
    # text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8')
    return text

def ingest_document(
    file_path: str,
    collection: chromadb.Collection,
    doc_type: str,
    metadata_extra: Optional[dict] = None,
    chunk_size: int = 500,
    overlap: int = 100,
) -> int:
    text  = load_document(file_path)
    text  = sanitize_text(text)
    words = text.split()
    chunks, start = [], 0
    while start < len(words):
        chunk = " ".join(words[start : start + chunk_size])
        if len(chunk) > 50:
            chunks.append(chunk)
        start += chunk_size - overlap

    doc_id    = Path(file_path).stem
    ids       = [f"{doc_id}_chunk_{i}" for i in range(len(chunks))]
    metadatas = [{
        "source": file_path, "doc_type": doc_type, "chunk_index": i,
        **(metadata_extra or {})
    } for i in range(len(chunks))]

    collection.add(documents=chunks, ids=ids, metadatas=metadatas)
    print(f"  ✔ '{file_path}' → {len(chunks)} chunks → [{collection.name}]")
    return len(chunks)

# ── Ingest all sample docs ────────────────────────────────────────────────────
print("Ingesting documents…")
ingest_document("roe_v_wade.txt",              CASE_LAW_COLLECTION,   "case_law",          {"jurisdiction": "US Federal", "year": "1973"})
ingest_document("employment_discrimination.txt", CASE_LAW_COLLECTION, "legal_brief",        {"jurisdiction": "US Federal", "year": "2024"})
ingest_document("gdpr_compliance.txt",         COMPLIANCE_COLLECTION, "compliance_policy",  {"jurisdiction": "EU", "effective_date": "2024-01-01"})
ingest_document("sec_ruling_2024.txt",         REGULATORY_COLLECTION, "regulatory_update",  {"regulator": "SEC", "effective_date": "2023-12-01"})

print("\n✅ All documents ingested and embedded")

Ingesting documents…
  ✔ 'roe_v_wade.txt' → 1 chunks → [case_law]
  ✔ 'employment_discrimination.txt' → 1 chunks → [case_law]
  ✔ 'gdpr_compliance.txt' → 1 chunks → [compliance]
  ✔ 'sec_ruling_2024.txt' → 1 chunks → [regulatory]

✅ All documents ingested and embedded


In [ ]:
# ─── CELL 5: Retrieval Engine ────────────────────────────────────────────────
def retrieve(
    query: str,
    collections: list,
    top_k: int = 5,
) -> list[dict]:
    all_results = []
    for col in collections:
        try:
            n = min(top_k, col.count())
            if n == 0:
                continue
            results = col.query(query_texts=[query], n_results=n)
            for i, doc in enumerate(results["documents"][0]):
                all_results.append({
                    "text":       doc,
                    "metadata":   results["metadatas"][0][i],
                    "distance":   results["distances"][0][i],
                    "collection": col.name,
                })
        except Exception as e:
            print(f"  Warning [{col.name}]: {e}")

    return sorted(all_results, key=lambda x: x["distance"])[:top_k]

def format_context(retrieved: list[dict]) -> str:
    lines = []
    for i, r in enumerate(retrieved, 1):
        src = r["metadata"].get("source", "unknown")
        dt  = r["metadata"].get("doc_type", "")
        lines.append(f"[{i}] Source: {src} | Type: {dt}\n{r['text']}")
    return "\n---\n".join(lines)

# ── Test retrieval ─────────────────────────────────────────────────────────────
hits = retrieve("privacy rights constitutional precedent",
                [CASE_LAW_COLLECTION, COMPLIANCE_COLLECTION])
print(f"Top {len(hits)} results:\n")
for h in hits:
    print(f"  [{h['collection']}] dist={h['distance']:.3f} | {h['text'][:100]}…\n")

Top 5 results:

  [case_law] dist=0.395 | CASE: Roe v. Wade (1973) Citation: 410 U.S. 113 Court: Supreme Court of the United States Decision: …

  [case_law] dist=0.654 | UNITED STATES DISTRICT COURT NORTHERN DISTRICT OF CALIFORNIA CASE NO: 2024-CV-08821 PLAINTIFF: DataS…

  [case_law] dist=0.704 | incident within 4 business days of determining it is material. ─────────────────────────────────────…

  [case_law] dist=0.736 | INTERNAL LEGAL BRIEF Matter: Johnson v. TechCorp Employment Discrimination Date: February 2024 Summa…

  [compliance] dist=0.767 | GDPR COMPLIANCE POLICY - Version 3.1 Effective Date: 2024-01-01 Regulatory Reference: EU Regulation …



In [ ]:
# ─── CELL 6: LLM Answer Generation with Citations ────────────────────────────
SYSTEM_PROMPT = """You are an expert legal research assistant.
Rules:
1. Cite every factual claim inline as [Source N].
2. Structure your answer: Summary -> Legal Analysis -> Key Precedents -> Caveats.
3. If the answer isn't in the context, say "Insufficient information in the knowledge base."
4. Never fabricate case names, statutes, or citations."""

def groq_chat(system: str, user: str, max_tokens: int = 2048) -> str:
    response = groq_client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": user},
        ],
        max_tokens=max_tokens,
        temperature=0.2,
    )
    return response.choices[0].message.content

def answer_legal_query(query: str, collections: Optional[list] = None, top_k: int = 5) -> dict:
    if collections is None:
        collections = [CASE_LAW_COLLECTION, COMPLIANCE_COLLECTION, REGULATORY_COLLECTION]

    retrieved = retrieve(query, collections, top_k=top_k)
    context   = format_context(retrieved)

    answer = groq_chat(
        system=SYSTEM_PROMPT,
        user=f"Context from legal database:\n\n{context}\n\n---\nLegal Query: {query}",
    )
    return {
        "query":   query,
        "answer":  answer,
        "sources": list({r["metadata"]["source"] for r in retrieved}),
    }

# ── Demo ───────────────────────────────────────────────────────────────────────
result = answer_legal_query("What are the precedents for employer liability and privacy rights?")
print(f"QUERY: {result['query']}\n")
print(f"SOURCES: {result['sources']}\n")
print(f"ANSWER:\n{result['answer']}")

QUERY: What are the precedents for employer liability and privacy rights?

SOURCES: ['roe_v_wade.txt', 'employment_discrimination.txt', '/content/legaldoc.pdf', '/content/legaldoc1.docx']

ANSWER:
**Summary**

The precedents for employer liability and privacy rights are rooted in various case laws and regulations. The key principles include the duty of care for employers to protect employee data, the right to privacy under the Due Process Clause, and the liability of employers for discriminatory practices.

**Legal Analysis**

Employer liability for privacy rights is established through several precedents:

1. **Dittman v. UPMC (2018)**: The Pennsylvania Supreme Court held that an employer that collects employee data has a duty to use reasonable care to protect that data from criminal intrusion [Source: /content/legaldoc.pdf | Type: case_law].
2. **In re: Equifax Data Breach Litigation (2019)**: The court held that companies storing sensitive consumer data owe a duty of care to impleme

In [ ]:
# ─── CELL 7: Regulatory Change Flagging ──────────────────────────────────────
def flag_regulatory_changes(topic: str, effective_after: str = "2023-01-01") -> dict:
    retrieved = retrieve(topic, [REGULATORY_COLLECTION], top_k=6)
    context   = format_context(retrieved)

    analysis = groq_chat(
        system="You are a legal compliance analyst. Be precise and structured.",
        user=f"""Analyze regulatory updates in the context below for the topic: '{topic}'.
Effective after: {effective_after}.

For each relevant update:
1. STATE the regulation name, regulator, and effective date.
2. EXPLAIN what changed and the compliance impact.
3. FLAG which documents or practices are affected.
4. ASSIGN urgency: IMMEDIATE / REVIEW IN 90 DAYS / MONITOR.

Context:
{context}""",
        max_tokens=1500,
    )
    return {"topic": topic, "analysis": analysis, "num_sources": len(retrieved)}

# ── Demo ───────────────────────────────────────────────────────────────────────
flags = flag_regulatory_changes("cybersecurity disclosure public companies")
print(f"REGULATORY FLAGS FOR: '{flags['topic']}'\n")
print(flags["analysis"])

REGULATORY FLAGS FOR: 'cybersecurity disclosure public companies'

**Regulatory Update Analysis: Cybersecurity Disclosure for Public Companies**

**Update 1: SEC Final Rule - Cybersecurity Risk Management, Strategy, Governance, and Incident Disclosure**

1. **Regulation Name:** Cybersecurity Risk Management, Strategy, Governance, and Incident Disclosure
2. **Regulator:** Securities and Exchange Commission (SEC)
3. **Effective Date:** December 2023

**What Changed and Compliance Impact:**

* Public companies must disclose material cybersecurity incidents within 4 business days.
* Annual disclosure of cybersecurity risk management processes is required in Form 10-K.
* Board oversight of cybersecurity risks must be documented.
* Non-disclosure is subject to SEC enforcement action.

**Affected Documents/Practices:**

* Form 10-K (annual report)
* Cybersecurity incident response plan
* Board meeting minutes and documentation
* Cybersecurity risk management policies and procedures

**Urgency

In [ ]:
# ─── CELL 8: Auto-Generate Legal Brief with Citations ────────────────────────
def generate_legal_brief(
    case_description: str, # the facts of the case, injected in the prompt
    client_position:  str, # what outcome the client wants
    jurisdiction:     str = "US Federal", # default parameter, tells the LLM which legal system's rules apply
) -> str:
    all_cols  = [CASE_LAW_COLLECTION, COMPLIANCE_COLLECTION, REGULATORY_COLLECTION]
    retrieved = retrieve(case_description, all_cols, top_k=6)
    context   = format_context(retrieved)

    brief = groq_chat( # the prompt engineering - 5 section structure
        system="You are a senior attorney drafting formal legal briefs. Be precise, structured, and cite every claim.",
        user=f"""Draft a formal legal brief for the following situation.

CASE DESCRIPTION: {case_description}
CLIENT POSITION: {client_position}
JURISDICTION: {jurisdiction}

Cite every legal assertion as [Source N]. Use ONLY the provided context.

Structure:
I.   STATEMENT OF FACTS
II.  LEGAL ISSUES PRESENTED
III. ARGUMENT (cite precedents for each sub-issue)
IV.  REGULATORY & COMPLIANCE CONSIDERATIONS
V.   CONCLUSION & RECOMMENDED ACTION

Research Context:
{context}""",
        max_tokens=3000,
    )
    # returns a str (complete document),
    # sources appended to the string rather than returned as a seperate field

    sources = list({r["metadata"]["source"] for r in retrieved})
    return brief + f"\n\n─── Sources consulted: {sources} ───"

# ── Demo ───────────────────────────────────────────────────────────────────────
brief = generate_legal_brief(
    case_description="Company suffered a data breach exposing 50,000 EU user records including health information.",
    client_position="Minimize regulatory penalties and demonstrate good-faith compliance efforts.",
)
print("═" * 60)
print("GENERATED LEGAL BRIEF")
print("═" * 60)
print(brief)

════════════════════════════════════════════════════════════
GENERATED LEGAL BRIEF
════════════════════════════════════════════════════════════
**IN THE UNITED STATES DISTRICT COURT FOR THE [DISTRICT]**

**DATA SAFE TECHNOLOGIES INC.,**

**Plaintiff,**

**v.**

**CLOUDSTORE SOLUTIONS LLC,**

**Defendant.**

**COMPLIANCE BRIEF**

**CASE NO. [CASE NUMBER]**

**JUNE 23, 2024**

**TO THE HONORABLE JUDGE OF THE [DISTRICT] COURT**

**STATEMENT OF FACTS**

1. On March 3, 2024, CloudStore Solutions LLC suffered a data breach that exposed personally identifiable information (PII) of approximately 50,000 EU user records, including health information.
2. DataSafe Technologies Inc. contracted CloudStore as a third-party data processor under a Data Processing Agreement (DPA) signed on January 15, 2023.
3. CloudStore failed to notify DataSafe of the breach until March 14, 2024, eleven days after discovery, in direct violation of GDPR Article 33 which mandates notification within 72 hours.

**LEGAL I

In [ ]:
# ─── CELL 9: Interactive Query Interface ─────────────────────────────────────
def legal_assistant(query: str, mode: str = "query") -> None:
    """
    mode options:
      'query'      — precedent / case law search with citations
      'regulatory' — flag regulatory changes for a topic
      'brief'      — auto-generate a full legal brief
    """
    print(f"\n{'═'*60}")
    print(f"MODE : {mode.upper()}")
    print(f"INPUT: {query}")
    print(f"{'═'*60}\n")

    if mode == "regulatory":
        out = flag_regulatory_changes(query)
        print(out["analysis"])

    elif mode == "brief":
        out = generate_legal_brief(
            case_description=query,
            client_position="Protect client interests while ensuring full regulatory compliance.",
        )
        print(out)

    else:  # default: query
        out = answer_legal_query(query)
        print(out["answer"])
        print(f"\n📚 Sources: {out['sources']}")

# ── Enter your questions here  ────────────────────────────────────────────────────────

legal_assistant("Employee was fired because of their race.", mode="brief")





════════════════════════════════════════════════════════════
MODE : BRIEF
INPUT: Employee was fired because of their race.
════════════════════════════════════════════════════════════

**IN THE UNITED STATES DISTRICT COURT FOR THE [DISTRICT]**

**JOHNSON v. TECHCORP EMPLOYMENT DISCRIMINATION**

**CASE NO. [CASE NUMBER]**

**BRIEF OF PLAINTIFF IN SUPPORT OF CLAIM FOR RELIEF UNDER TITLE VII OF THE CIVIL RIGHTS ACT OF 1964**

**I. STATEMENT OF FACTS**

1. Plaintiff, [Johnson], was employed by TechCorp as a [position] from [date] to [date].
2. On [date], Plaintiff was terminated from employment by TechCorp.
3. Plaintiff alleges that the termination was based on his race, in violation of Title VII of the Civil Rights Act of 1964.
4. TechCorp has denied any wrongdoing and asserts that the termination was based on legitimate, non-discriminatory reasons.

**II. LEGAL ISSUES PRESENTED**

1. Whether TechCorp's termination of Plaintiff was based on his race, in violation of Title VII of the Civi

In [ ]:
# ─── CELL 10: Add Your Own Documents ─────────────────────────────────────────
# Upload any PDF or TXT to Colab, then call one of these helpers:

def add_case_law(file_path: str, jurisdiction: str, year: str):
    ingest_document(file_path, CASE_LAW_COLLECTION, "case_law",
                    {"jurisdiction": jurisdiction, "year": year})

def add_compliance_policy(file_path: str, effective_date: str):
    ingest_document(file_path, COMPLIANCE_COLLECTION, "compliance_policy",
                    {"effective_date": effective_date})

def add_regulatory_update(file_path: str, regulator: str, effective_date: str):
    ingest_document(file_path, REGULATORY_COLLECTION, "regulatory_update",
                    {"regulator": regulator, "effective_date": effective_date})

# Examples (upload your file first via the Colab file browser on the left):
# add_case_law("my_case.pdf",        jurisdiction="California", year="2024")
# add_compliance_policy("policy.pdf", effective_date="2024-06-01")
# add_regulatory_update("ftc_rule.pdf", regulator="FTC", effective_date="2024-03-15")

print("✅ Ready. Upload your documents and call the helpers above.")
print()
print("Then query with:")
print('  legal_assistant("your question",    mode="query")')
print('  legal_assistant("topic to scan",    mode="regulatory")')
print('  legal_assistant("your case facts",  mode="brief")')

✅ Ready. Upload your documents and call the helpers above.

Then query with:
  legal_assistant("your question",    mode="query")
  legal_assistant("topic to scan",    mode="regulatory")
  legal_assistant("your case facts",  mode="brief")


In [ ]:
# ─── CELL 11: Test All Features ──────────────────────────────────────────────

print("=" * 60)
print("TEST 1: Precedent Search")
print("=" * 60)
legal_assistant(
    "What are the legal precedents for racial discrimination in employment?",
    mode="query"
)

print("\n" + "=" * 60)
print("TEST 2: Privacy & Data Rights Query")
print("=" * 60)
legal_assistant(
    "What are the rules around data breach notification and penalties?",
    mode="query"
)

print("\n" + "=" * 60)
print("TEST 3: Regulatory Change Flagging")
print("=" * 60)
legal_assistant(
    "cybersecurity disclosure requirements for public companies",
    mode="regulatory"
)

print("\n" + "=" * 60)
print("TEST 4: Auto-Generate Legal Brief")
print("=" * 60)
legal_assistant(
    "A tech company exposed 10,000 EU customer records in a data breach. "
    "The company delayed notifying authorities for 2 weeks.",
    mode="brief"
)

TEST 1: Precedent Search

════════════════════════════════════════════════════════════
MODE : QUERY
INPUT: What are the legal precedents for racial discrimination in employment?
════════════════════════════════════════════════════════════

**Summary**
The legal precedents for racial discrimination in employment are rooted in Title VII of the Civil Rights Act of 1964, which prohibits employment discrimination based on race, color, religion, sex, or national origin. Key precedents include:

*   **Griggs v. Duke Power Co. (1971)**: Established the disparate impact theory, which allows plaintiffs to prove discrimination through statistical evidence showing a neutral employment practice has a disproportionate adverse effect on a protected class.
*   **McDonnell Douglas Corp. v. Green (1973)**: Introduced the burden-shifting framework for disparate treatment claims, requiring plaintiffs to establish a prima facie case, followed by the employer's burden to show a legitimate reason, and finall

In [ ]:
# ─── Ingest My PDF ───────────────────────────────────────────────────────────
add_case_law(
    "/content/legaldoc.pdf",   # replace with your exact filename
    jurisdiction="US Federal",
    year="2024"
)

  ✔ '/content/legaldoc.pdf' → 2 chunks → [case_law]


In [ ]:
legal_assistant(
   "What is CloudStore's liability for the data breach notification delay?",
      mode="query"
 )


════════════════════════════════════════════════════════════
MODE : QUERY
INPUT: What is CloudStore's liability for the data breach notification delay?
════════════════════════════════════════════════════════════

**Summary**
CloudStore Solutions LLC is facing a data breach liability lawsuit from DataSafe Technologies Inc. due to a data breach that exposed personally identifiable information (PII) of approximately 120,000 users. The breach occurred on March 3, 2024, and CloudStore failed to notify DataSafe until March 14, 2024, eleven days after discovery, in direct violation of GDPR Article 33 which mandates notification within 72 hours.

**Legal Analysis**
CloudStore's liability for the data breach notification delay can be analyzed under GDPR Article 33, which requires controllers to notify the supervisory authority of a personal data breach within 72 hours of becoming aware of it. In this case, CloudStore failed to notify DataSafe within the required timeframe, which may constitut

In [ ]:
# ─── DOCX Support ────────────────────────────────────────────────────────────
!pip install -q python-docx

from docx import Document as DocxDocument

def extract_text_from_docx(file_path: str) -> str:
    """Extract all text from a Word document."""
    doc   = DocxDocument(file_path)
    lines = []

    for para in doc.paragraphs:
        if para.text.strip():
            lines.append(para.text.strip())

    # Also extract text from tables inside the document
    for table in doc.tables:
        for row in table.rows:
            for cell in row.cells:
                if cell.text.strip():
                    lines.append(cell.text.strip())

    return "\n".join(lines)

# ── Patch load_document() to support .docx ───────────────────────────────────
def load_document(file_path: str) -> str:
    ext = Path(file_path).suffix.lower()
    if ext == ".pdf":
        return extract_text_from_pdf(file_path)
    elif ext in (".txt", ".md"):
        return Path(file_path).read_text(encoding="utf-8", errors="ignore")
    elif ext in (".docx", ".doc"):
        return extract_text_from_docx(file_path)
    else:
        raise ValueError(f"Unsupported file type: {ext}")

# ── Helper functions for Word documents ──────────────────────────────────────
def add_case_law_docx(file_path: str, jurisdiction: str, year: str):
    ingest_document(file_path, CASE_LAW_COLLECTION, "case_law",
                    {"jurisdiction": jurisdiction, "year": year})

def add_compliance_policy_docx(file_path: str, effective_date: str):
    ingest_document(file_path, COMPLIANCE_COLLECTION, "compliance_policy",
                    {"effective_date": effective_date})

def add_regulatory_update_docx(file_path: str, regulator: str, effective_date: str):
    ingest_document(file_path, REGULATORY_COLLECTION, "regulatory_update",
                    {"regulator": regulator, "effective_date": effective_date})

print("✅ DOCX support added. All file types now supported:")
print("   .txt  → plain text")
print("   .pdf  → PyPDF2 extraction")
print("   .docx → python-docx extraction")

✅ DOCX support added. All file types now supported:
   .txt  → plain text
   .pdf  → PyPDF2 extraction
   .docx → python-docx extraction


In [ ]:
# ─── CELL 12: Install LangChain Dependencies ──────────────────────────────────
!pip install -q langchain langchain-community langchain-groq

In [ ]:
# ─── CELL 13: LangChain Setup (Embeddings & LLM) ─────────────────────────────
from langchain_groq import ChatGroq
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains import RetrievalQA
from langchain_core.prompts import ChatPromptTemplate

# Initialize the LLM via LangChain
llm = ChatGroq(groq_api_key=os.environ["GROQ_API_KEY"], model_name=MODEL, temperature=0.2)

# Initialize Embeddings (matching your previous model)
hf_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Initialize LangChain's Chroma wrapper using the existing collections
# Note: LangChain typically manages one collection per vectorstore object
vectorstore = Chroma(
    collection_name="case_law",
    embedding_function=hf_embeddings,
    persist_directory="./chroma_db" # Optional persistence
)

print("✅ LangChain LLM and VectorStore wrappers ready")

In [ ]:
# ─── CELL 14: LangChain Ingestion Pipeline ────────────────────────────────────
from langchain_community.document_loaders import PyPDFLoader, TextLoader, Docx2txtLoader

def ingest_with_langchain(file_path: str):
    ext = Path(file_path).suffix.lower()

    # Select loader
    if ext == ".pdf": loader = PyPDFLoader(file_path)
    elif ext == ".docx": loader = Docx2txtLoader(file_path)
    else: loader = TextLoader(file_path)

    # Load and Split
    docs = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    splits = text_splitter.split_documents(docs)

    # Add to VectorStore
    vectorstore.add_documents(splits)
    print(f"✅ Ingested {len(splits)} chunks from {file_path} via LangChain")

# Example usage:
ingest_with_langchain("roe_v_wade.txt")

In [ ]:
# ─── CELL 15: LangChain RAG Chain ─────────────────────────────────────────────
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

# Define a custom prompt template
prompt = ChatPromptTemplate.from_template("""
You are an expert legal research assistant. Use the following pieces of retrieved context to answer the query.
If you don't know the answer, say that you don't know.

Context: {context}

Question: {input}

Answer:""")

# Create the retrieval chain
document_chain = create_stuff_documents_chain(llm, prompt)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
rag_chain = create_retrieval_chain(retriever, document_chain)

# Test the chain
response = rag_chain.invoke({"input": "What was the core decision in Roe v. Wade?"})
print(response["answer"])

In [ ]:
add_case_law_docx(
    "/content/legaldoc1.docx",
    jurisdiction="US Federal",
    year="2024"
)

  ✔ '/content/legaldoc1.docx' → 3 chunks → [case_law]


In [ ]:
legal_assistant(
    "What is the evidence for pay discrimination at NexaCorp?",
    mode="query"
)


════════════════════════════════════════════════════════════
MODE : QUERY
INPUT: What is the evidence for pay discrimination at NexaCorp?
════════════════════════════════════════════════════════════

**Summary**
The evidence for pay discrimination at NexaCorp includes statistical data, internal admissions, and witness testimony. The plaintiffs in the Rivera et al. class action lawsuit against NexaCorp have presented the following evidence:

* Statistical data showing a 34% pay gap between men and women in equivalent roles across 840 hiring decisions [Source: /content/legaldoc1.docx]
* Internal 2022 compensation analysis acknowledging the pay disparity [Source: /content/legaldoc1.docx]
* Three HR manager witnesses willing to testify that they were verbally instructed to prioritize white male candidates for senior positions [Source: /content/legaldoc1.docx]
* A 3x rejection rate for minority candidates with equal qualifications [Source: /content/legaldoc1.docx]

**Legal Analysis**
The e